# cl-mma-fondos — extraction & clean (release v1, full MMA set)

Source: MMA competitive funds portal `https://fondos.mma.gob.cl/` (FPA + FPR + Recambio). Review README: `../../README.md`. Need: `2026-06-cl-finance-opportunities`.
Two fidelity tiers, one row per fund **line**: **detailed** (full Antecedentes parsed for the current cycle) and **index** (name/URL/open-date/status enumerated from the FPA + FPR hubs, 2020–2025 archive). Field values captured from the live portal on 2026-06-08. Restart-and-run-all passes.

## Helpers — each implements one README parsing note (amount/date/actor/gpc/status).

In [1]:
import re, json, datetime as dt
import pandas as pd
AS_OF = dt.date(2026,6,8)
BASE = "https://fondos.mma.gob.cl/"
SPAN={"enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,"julio":7,"agosto":8,
"septiembre":9,"setiembre":9,"octubre":10,"noviembre":11,"diciembre":12}
def amt(t):
    if not t: return None,False
    groups=[g for g in re.split(r"[.\s]", re.sub(r"[^\d.]","",t)) if g]
    suspect = any(len(g)!=3 for g in groups[1:]) if len(groups)>1 else False
    d=re.sub(r"[^\d]","",t); return (int(d) if d else None), suspect
def dmy(t):
    if not t: return None
    t=t.lower()
    m=re.search(r"(\d{1,2})\s+de\s+([a-záéíóú]+)\s+de[l]?\s+(\d{4})",t)
    if m and SPAN.get(m.group(2)): return dt.date(int(m.group(3)),SPAN[m.group(2)],int(m.group(1))).isoformat()
    m=re.search(r"(\d{1,2})[/-](\d{1,2})[/-](\d{4})",t)
    if m: return dt.date(int(m.group(3)),int(m.group(2)),int(m.group(1))).isoformat()
    return None
def actor(t):
    t=(t or "").lower()
    if "municipalidad" in t: return "municipality"
    if "indígena" in t or "indigena" in t or "conadi" in t: return "indigenous community"
    if "centro de investigaci" in t or "universidad" in t: return "research/university + ngo"
    if any(k in t for k in ["sin fines de lucro","junta","ong","fundaci","corporaci","centros de padres","organizaci","comunitaria"]): return "community/citizen org"
    return "unspecified"
LG=[(r"cambio clim|descontaminaci","cross_sector"),(r"econom.a circular|residuo|recicl","waste"),
(r"eficiencia energ|energ.a renovable|solar|fotovolta","stationary_energy"),
(r"biodivers|conservaci|fauna|humedal|criosfera|marino|ecosistema|área verde|area verde|pudú|pudu|incendio","afolu"),
(r"h.dric|agua","water")]
def gpc(text):
    out=[]
    for pat,sec in LG:
        if re.search(pat,(text or "").lower()) and sec not in out: out.append(sec)
    return out or ["cross_sector"]
def status_for(close_iso, lifecycle):
    if close_iso:
        st="open" if dt.date.fromisoformat(close_iso)>=AS_OF else "closed"
        return st,(st=="closed" and lifecycle=="open_call")
    return ("closed" if lifecycle in ("awarded",) else "unknown"), False

## Detailed tier — current-cycle fichas (real captured values; raw strings kept so cleaning is auditable).

In [2]:
# ---------- DETAILED (7 funds, real captured values) ----------
LINEAS5="Ecotecnias Hídricas; Cambio Climático y Descontaminación Ambiental; Economía Circular y Gestión de Residuos; Valoración y Conservación de la Biodiversidad; Eficiencia Energética y Energías Renovables"
DETAILED=[
 dict(slug="fpa-2026-proyectos-sustentables-ciudadanos",program="FPA",name="FPA 2026 - Proyectos Sustentables Ciudadanos",
   tipo="personas jurídicas de derecho privado y sin fines de lucro (Juntas de Vecinos, ONG, Fundaciones, Corporaciones)",
   fin="$ 6.000.000",lineas=LINEAS5,inicio="26 de agosto del 2025",cierre="07 de octubre del 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/08/Resolucion-Exenta-5796-Aprueba-Bases-FPA-2026-Proyectos-Sustentables-Ciudadanos.pdf"),
 dict(slug="fpa-2026-proyectos-sustentables-en-establecimientos-educacionales",program="FPA",name="FPA 2026 - Proyectos Sustentables en Establecimientos Educacionales",
   tipo="Centros de Padres y Apoderados con personalidad jurídica de derecho privado y sin fines de lucro",
   fin="$ 6.000.000",lineas=LINEAS5,inicio="26 de agosto del 2025",cierre="07 de octubre del 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/08/Resolucion-Exenta-5797-Aprueba-Bases-FPA-2026-Proyectos-Sustentables-Establecimientos-Educacionales.pdf"),
 dict(slug="fpa-2026-proyectos-sustentables-para-pueblos-indigenas",program="FPA",name="FPA 2026 - Proyectos Sustentables para Pueblos Indígenas",
   tipo="Comunidades o Asociaciones Indígenas inscritas en CONADI",
   fin="$ 6.000.000",lineas=LINEAS5,inicio="26 de agosto del 2025",cierre="07 de octubre del 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/08/Resolucion-Exenta-5798-Aprueba-Bases-FPA-2026-Proyectos-Sustentables-Pueblos-Indigenas.pdf"),
 dict(slug="fpa-2026-rapanui",program="FPA",name="FPA Extraordinario Rapa Nui 2026",
   tipo="Comunidades o Asociaciones Indígenas inscritas en CONADI",
   fin="$ 10.000.0000",lineas="Protección y/o conservación del medio marino de Rapa Nui; Protección y/o conservación de la biodiversidad de Rapa Nui",
   inicio="02 de octubre del 2025",cierre="24 de octubre del 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/10/RES-6732-Aprueba-Bases-Especiales-Concurso-Extraordinario-Rapa-Nui-2026.pdf"),
 dict(slug="altodelcarmen-hidrico-2025",program="FPA",name="FPA 2025 - Alto del Carmen: Ecosistemas Altoandinos – Humedales y Criósfera",
   tipo="Fundaciones, Corporaciones y ONG; Universidades; Centros de Investigación",
   fin="$70.000.000",lineas="Humedales y criósfera; ecosistemas altoandinos; gestión sustentable",inicio="05 de agosto del 2025",cierre="17 de septiembre de 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/08/Resolucion_Ex_No5164-2025_Aprueba_Bases_Concurso_FPA_Puesta_en_valor_de_los_Ecosistemas_Altoandinos_Humedales_y_Criosfera.pdf"),
 dict(slug="altodelcarmen-biodiversidad-2025",program="FPA",name="FPA 2025 - Alto del Carmen: Ecosistemas Altoandinos – Biodiversidad",
   tipo="Fundaciones, Corporaciones y ONG; Universidades; Centros de Investigación",
   fin="$60.000.000",lineas="Biodiversidad; ecosistemas altoandinos; conservación efectiva",inicio="05 de agosto del 2025",cierre="17 de septiembre de 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/08/Resolucion_Ex_No5165-2025_Aprueba_Bases_Concurso_FPA_Puesta_en_valor_de_los_Ecosistemas_Altoandinos_Biodiversidad.pdf"),
 dict(slug="fpr-2026-fondo-para-el-reciclaje",program="FPR",name="FPR 2026 - Fondo para el Reciclaje",
   tipo="Municipalidades y Asociaciones de Municipalidades inscritas en el Registro Único de Asociaciones Municipales de SUBDERE",
   fin="$ 14.000.000",lineas="Economía Circular y Gestión de Residuos; Ley REP; recicladores de base; educación ambiental",inicio="24 de septiembre del 2025",cierre="30 de octubre del 2025",
   res="https://fondos.mma.gob.cl/wp-content/uploads/2025/09/RES.-EX.-N6534-2025_APRUEBA-BASES-FPR-2026.pdf"),
]
rows=[]
for d in DETAILED:
    a,susp=amt(d["fin"]); close=dmy(d["cierre"]); st,conf=status_for(close,"awarded")
    rows.append(dict(fund_program=d["program"],fund_name=d["name"],eligible_actor=actor(d["tipo"]),
      amount_clp=a,amount_suspect=susp,instrument_type="grant",open_date=dmy(d["inicio"]),close_date=close,
      status=st,lifecycle="awarded",status_section_conflict=conf,status_as_of=AS_OF.isoformat(),
      specificity="sector-specific",climate_relevance="explicit",
      gpc_sectors=json.dumps(gpc(d["lineas"]+" "+d["name"]),ensure_ascii=False),
      thematic_lines=json.dumps([x.strip() for x in d["lineas"].split(";") if x.strip()],ensure_ascii=False),
      eligible_actor_detail=d["tipo"],access_pathway="direct application (via fondos.gob.cl)",
      detail_level="detailed",resolucion_url=d["res"],source_url=BASE+d["slug"]+"/"))
print(len(rows),'detailed rows')

7 detailed rows


## Index tier — enumerated from FPA + FPR hubs (breadth). Reads `data/hub_index.json` (committed).
Re-capture the hubs to refresh; see the live-refresh helper at the end.

In [3]:
import json
INDEX=[(d['program'],d['name'],d['slug'],d['open_raw']) for d in json.load(open('data/hub_index.json'))]
for prog,name,slug,opn in INDEX:
    actor_guess = "municipality" if prog=="FPR" else ("household" if "calefactor" in slug else actor(name))
    rows.append(dict(fund_program=prog,fund_name=name,eligible_actor=actor_guess,amount_clp=None,amount_suspect=False,
      instrument_type=("grant" if prog!="PROG" else "subsidy"),open_date=dmy(opn or ""),close_date=None,
      status="closed",lifecycle="awarded",status_section_conflict=False,status_as_of=AS_OF.isoformat(),
      specificity="sector-specific",climate_relevance="explicit",gpc_sectors=json.dumps(gpc(name),ensure_ascii=False),
      thematic_lines="[]",eligible_actor_detail=None,access_pathway="direct application (via fondos.gob.cl)",
      detail_level="index",resolucion_url=None,source_url=BASE+slug+"/"))
print(len(INDEX),'index rows')

48 index rows


## Assemble

In [4]:
df=pd.DataFrame(rows)
df[['fund_program','detail_level']].value_counts()

fund_program  detail_level
FPA           index           39
FPR           index            8
FPA           detailed         6
FPR           detailed         1
PROG          index            1
Name: count, dtype: int64

## Recurrence — classify each line annual / sporadic / one-off from its presence across 2020-2026 cycles; set a next-call estimate for annual streams.

In [5]:
# ---------- RECURRENCE (cross-cycle pattern across the 2020-2026 archive) ----------
def stream(n):
    n=n.lower()
    # specific / special calls first (their titles often also contain generic words like "ciudadanas")
    if "reciclaje" in n or "recicladores" in n or "economía circular en municipios" in n: return "FPR Reciclaje"
    if "calefactor" in n: return "Recambio Calefactores"
    if "tsej" in n or "transición socioecológica" in n: return "FPA TSEJ (zonas)"
    if "alto del carmen" in n or "altoandino" in n: return "FPA Alto del Carmen"
    if "rapa nui" in n: return "FPA Rapa Nui"
    if "fauna" in n or "pudú" in n or "pudu" in n or "rehabilitación" in n or "incendios" in n: return "FPA Fauna/biodiv especial"
    if "áreas verdes" in n or "areas verdes" in n or "humedales" in n or "marina" in n: return "FPA Areas verdes/Humedales"
    if "fundaciones y corporaciones" in n: return "FPA Fundaciones/Corp"
    # generic recurring streams
    if "establecimientos educacionales" in n or "chiloé reduce" in n: return "FPA Educacionales"
    if "indígena" in n or "indigena" in n: return "FPA Pueblos Indígenas"
    if "ciudadan" in n: return "FPA Ciudadanos"
    return "FPA Otros tematicos"
# recurrence grounded in how many of the 7 cycles (2020-2026) each stream appears in (see review.md)
NEXT="~ago-oct 2026 (cycle 2027)"
RECUR={
 "FPR Reciclaje":("annual",NEXT),
 "FPA Ciudadanos":("annual",NEXT),
 "FPA Educacionales":("annual",NEXT),
 "FPA Pueblos Indígenas":("annual",NEXT),
 "FPA Rapa Nui":("sporadic",None),
 "FPA Areas verdes/Humedales":("sporadic",None),
 "FPA Fundaciones/Corp":("sporadic",None),
 "FPA Otros tematicos":("sporadic",None),
 "FPA Alto del Carmen":("one-off",None),
 "FPA TSEJ (zonas)":("one-off",None),
 "FPA Fauna/biodiv especial":("one-off",None),
 "Recambio Calefactores":("ongoing (rolling, per-comuna PDA)","rolling"),
}
df["stream"]=df.fund_name.apply(stream)
df["recurrence"]=df.stream.map(lambda x: RECUR[x][0])
df["next_call_estimate"]=df.stream.map(lambda x: RECUR[x][1])
df.groupby('recurrence').fund_name.count()

recurrence
annual                               27
one-off                              11
ongoing (rolling, per-comuna PDA)     1
sporadic                             16
Name: fund_name, dtype: int64

## Validate — assertions, not eyeballing. Rapa Nui amount typo asserted as a NAMED source defect.

In [6]:
# ---------- VALIDATE ----------
assert len(df)==len(DETAILED)+len(INDEX)==55, len(df)
assert df["source_url"].str.startswith("https://fondos.mma.gob.cl/").all()
assert df["source_url"].is_unique, "dup urls"
d=df[df.detail_level=="detailed"].set_index("fund_name")
assert d.loc["FPR 2026 - Fondo para el Reciclaje","amount_clp"]==14000000
assert d.loc["FPR 2026 - Fondo para el Reciclaje","eligible_actor"]=="municipality"
assert d.loc["FPA 2026 - Proyectos Sustentables Ciudadanos","amount_clp"]==6000000
# named source-side defect: Rapa Nui amount string malformed ("$ 10.000.0000")
rn=d.loc["FPA Extraordinario Rapa Nui 2026"]
assert rn["amount_suspect"]==True, "Rapa Nui amount typo should be flagged"
assert df[df.amount_suspect]["fund_name"].tolist()==["FPA Extraordinario Rapa Nui 2026"], "only known defect allowed"
assert (df["status"]=="closed").all(), "all listed cycles closed as of AS_OF"
assert df["recurrence"].notna().all(), "recurrence must be set for every row"
ann=df[df.recurrence=="annual"].stream.unique().tolist()
assert set(ann)=={"FPR Reciclaje","FPA Ciudadanos","FPA Educacionales","FPA Pueblos Indígenas"}, ann
assert df[df.recurrence=="annual"]["next_call_estimate"].notna().all(), "annual rows need a next-call estimate"
assert set(df[df.fund_name.str.contains("Alto del Carmen|TSEJ")].recurrence)=={"one-off"}, "special calls = one-off"
print("ALL ASSERTIONS PASSED — rows:",len(df))
print("by program:\n", df.fund_program.value_counts().to_string())
print("by detail_level:\n", df.detail_level.value_counts().to_string())
print("eligible_actor:\n", df.eligible_actor.value_counts().to_string())

ALL ASSERTIONS PASSED — rows: 55
by program:
 fund_program
FPA     45
FPR      9
PROG     1
by detail_level:
 detail_level
index       48
detailed     7
eligible_actor:
 eligible_actor
unspecified                  29
municipality                  9
community/citizen org         7
indigenous community          7
research/university + ngo     2
household                     1


## Export — tidy table to `data/` (committed).

In [7]:
df.to_csv("data/cl_mma_fondos_v1.csv",index=False)
json.dump([dict(program=p,name=n,slug=s,open_raw=o) for p,n,s,o in INDEX], open("data/hub_index.json","w"),ensure_ascii=False,indent=1)
print("\nwrote data/cl_mma_fondos_v1.csv", df.shape, "+ data/hub_index.json")


wrote data/cl_mma_fondos_v1.csv (55, 24) + data/hub_index.json


## Fit analysis — vs need broad-capture must-haves + attribute coverage.

In [8]:
mh={"live source_url":df.source_url.str.startswith("https://").all(),
    "verifiable status+as_of":df.status.notna().all() and df.status_as_of.notna().all(),
    "identifiable funder":df.fund_program.notna().all()}
print("MUST-HAVES:",{k:("PASS" if v else "FAIL") for k,v in mh.items()})
print("detail-tier attribute coverage:")
det=df[df.detail_level=="detailed"]
for c in ["eligible_actor","amount_clp","close_date","gpc_sectors"]:
    print(f"  {det[c].notna().mean():.0%}  {c}")
print("broad funds present?:", (df.specificity=="broad").any(), "(MMA = all sector-specific)")

MUST-HAVES: {'live source_url': 'PASS', 'verifiable status+as_of': 'PASS', 'identifiable funder': 'PASS'}
detail-tier attribute coverage:
  100%  eligible_actor
  100%  amount_clp
  100%  close_date
  100%  gpc_sectors
broad funds present?: False (MMA = all sector-specific)


## Findings

- **Volume**: 55 MMA fund lines captured (45 FPA, 9 FPR, 1 Recambio programme); 7 detailed + 48 index. This is the realistic MMA universe — far beyond the 2-fund proof-of-concept.
- **Source defect (named)**: the FPA Rapa Nui ficha lists `$ 10.000.0000` (an extra zero); flagged via `amount_suspect`, not silently parsed to $100M. Intended value is $10,000,000 — confirm from the Bases PDF before use.
- **Amount spread is real**: FPA citizen lines $6M; FPR $14M; Alto del Carmen special calls $60–70M — so "FPA" is not one amount, reinforcing per-line modelling.
- **Eligible actor varies by line**: municipalities (FPR), community/citizen orgs (FPA ciudadanos/educacionales), indigenous communities (CONADI lines), research+NGO (Alto del Carmen). Captured per row.
- **All listed cycles are closed** as of 2026-06-08 (annual funds between cycles). Index rows carry name/URL/open-date for recurrence analysis; detailed rows add amount/actor/dates/sectors.
- **To deepen**: enrich index rows by fetching their fichas; pull the FPA hub `?page=2+` for pre-2020; parse Bases PDFs for exact eligibility/regional limits.